# Step 3E: evaluate the proportional 200-ad pilot

This notebook reads saved files only; it never calls an API. The sample excludes the 30 policy-development advertisements and uses proportional quotas from the 5,023 collected candidates. Its eligible frame is 4,993 ads. This is not complete Swedish demand. The old 30-ad balanced pilot and assistant review are not ground truth.

Set `RUN_DIR` to a saved Step 3E run after explicit paid execution. Initially it points to the dry run; no fictitious model classifications or accuracy figures are generated.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
PREPARED = ROOT / 'data/processed/step3e-prepared-20261022/sample'
RUN_DIR = ROOT / 'data/processed/step3e-prepared-20261022/dry-run'
manifest = json.loads((PREPARED / 'sampling_manifest.json').read_text(encoding='utf-8'))
sample = pd.read_csv(PREPARED / 'representative_200.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
summary = json.loads((RUN_DIR / 'run_summary.json').read_text(encoding='utf-8'))
rows = [json.loads(line) for line in (RUN_DIR / 'pilot_results.jsonl').read_text(encoding='utf-8').splitlines()]
results = pd.DataFrame(rows)
assert sample.advertisement_id.nunique() == len(sample) == 200
assert not set(sample.advertisement_id).intersection(manifest['excluded_ids'])
assert results.advertisement_id.is_unique
assert set(results.advertisement_id) <= set(sample.advertisement_id)
accepted = results.loc[results.ai_status.isin(['completed', 'cache_hit'])].copy()
display(pd.DataFrame(manifest['sampling_breakdown']))
print(manifest['sample_scope'])
print('Saved mode:', summary['mode'], '| Prompt:', summary['prompt_version'])

## Technical coverage and classification distributions

Technical acceptance means valid structured, source-grounded output, not correctness. Missing descriptions, failed responses, stopped requests and dry-run rows are unclassified operational outcomes; do not count them as irrelevant or model-uncertain. Partial runs have incomplete coverage. Distributions below include accepted model predictions only.

In [ ]:
display(pd.DataFrame([{'selected_in_run': len(results), 'accepted': len(accepted),
    'unclassified': len(results) - len(accepted),
    'technical_success_rate': summary['technical_success_rate'],
    'review_required_all_rows': summary['review_required']}]))
display(results.ai_status.value_counts().rename_axis('operational_status').to_frame('count'))
if accepted.empty:
    print('No accepted model predictions exist in this saved run. A dry run is not a quality measurement.')
else:
    display(accepted.ai_relevance.value_counts().rename_axis('AI relevance').to_frame('count'))
    display(accepted.ai_primary_category.fillna('(null)').value_counts().rename_axis('AI category').to_frame('count'))
    display(accepted.groupby(['ai_role_scope', 'ai_is_internship'], dropna=False).size().to_frame('count'))

## Disagreements and previously uncertain cases

Comparisons use the rule-based predictions as a comparator, never ground truth. A rule-uncertain prediction becoming resolved is a disagreement, not proof of improvement. Null-to-category resolutions are distinguished from competing nonnull categories. Failed responses do not enter these denominators.

In [ ]:
if accepted.empty:
    print('Disagreements and uncertain resolutions are unavailable until model predictions exist.')
else:
    rel_diff = accepted.relevance.ne(accepted.ai_relevance)
    cat_diff = accepted.primary_category.fillna('').ne(accepted.ai_primary_category.fillna(''))
    resolved = accepted.relevance.eq('uncertain') & accepted.ai_relevance.isin(['relevant', 'irrelevant'])
    both_categories = accepted.primary_category.fillna('').ne('') & accepted.ai_primary_category.fillna('').ne('')
    display(pd.DataFrame([{'comparison_denominator': len(accepted), 'relevance_disagreements': int(rel_diff.sum()),
        'category_disagreements': int(cat_diff.sum()),
        'nonnull_category_switches': int((cat_diff & both_categories).sum()),
        'previously_uncertain_resolved': int(resolved.sum())}]))
    display(accepted.loc[rel_diff | cat_diff, ['advertisement_id', 'job_title', 'relevance', 'primary_category',
        'ai_relevance', 'ai_primary_category', 'ai_reasoning', 'ai_evidence']])

## Evidence, confidence and cases requiring inspection

Inspect full duties and requirements. Low confidence, multi-role candidates and normalized whitespace matches warrant review. The original quotation, exact original source span and character offsets are retained; no paraphrases are accepted. Read failed-response diagnostics from the saved checkpoint, not as accepted predictions.

In [ ]:
if not accepted.empty:
    needs_review = accepted.ai_review_required.eq(True) | accepted.ai_confidence.eq('low')
    columns = [c for c in ['advertisement_id', 'job_title', 'ai_relevance', 'ai_primary_category', 'ai_confidence',
        'ai_multi_role_review_candidate', 'ai_reasoning', 'ai_evidence', 'ai_evidence_method',
        'ai_evidence_source_field', 'ai_evidence_start', 'ai_evidence_end', 'ai_evidence_source_excerpt'] if c in accepted]
    display(accepted.loc[needs_review, columns])
    print('Read descriptions with results.loc[results.advertisement_id.eq(JOB_ID), "description_text"].iloc[0]')
print('Operationally unclassified rows:', int((~results.ai_status.isin(['completed', 'cache_hit'])).sum()))

## Actual reported tokens, token-priced cost estimates and reservations

Provider-reported token usage is measured API usage; pricing it at configured rates is an estimate, not an invoice. Failed model answers can consume tokens. Conservative holds for unreported usage are separate. A local response cache hit incurs no new request; provider prompt-caching discounts are not separately reconciled. Forecasts are not measured usage.

In [ ]:
display(pd.DataFrame([{key: summary[key] for key in ['provider_reported_input_tokens',
    'provider_reported_output_tokens', 'provider_reported_usage_estimated_cost_usd',
    'unreported_usage_reserved_usd', 'accounted_cost_usd', 'new_attempts_reserved', 'budget_usd']}]))
display(pd.DataFrame([summary['forecast']]))
checkpoint_path = RUN_DIR / 'checkpoint.json'
checkpoint = json.loads(checkpoint_path.read_text(encoding='utf-8')) if checkpoint_path.exists() else {'ledger': []}
ledger = pd.DataFrame(checkpoint['ledger'])
if not ledger.empty:
    display(ledger[['advertisement_id', 'status', 'prompt_tokens', 'completion_tokens',
        'usage_basis', 'accounted_cost_usd', 'reserved_cost_usd']])

## Results by sampling stratum

Group sizes reflect rule relevance prevalence, not artificial balance. Strata weights describe the eligible candidate frame; the excluded development ads have zero inclusion probability. Raw model distributions describe this sample. For eventual design-based estimates, use recorded inclusion probabilities/weights and handle nonresponse explicitly. Never pool the separate targeted 30-ad diagnostics with these metrics unweighted.

In [ ]:
display(pd.DataFrame(summary['by_sampling_group']))
if not accepted.empty:
    display(pd.crosstab(accepted.sampling_group, accepted.ai_relevance, dropna=False))
if not ledger.empty:
    grouped_usage = ledger.merge(sample[['advertisement_id', 'sampling_group']], on='advertisement_id', validate='many_to_one')
    display(grouped_usage.groupby('sampling_group')[['prompt_tokens', 'completion_tokens', 'accounted_cost_usd']].sum())
print('Separate targeted diagnostic CSV:', PREPARED / 'targeted_diagnostic_30.csv')

## Independent human review ? labels start blank

Use `independent_human_review.csv`. Read full advertisements without treating rule or AI predictions as reference answers. Enter human relevance/category/notes, reviewer identity and explicit verification only after checking. Assistant opinions and generated labels must never be promoted to verified human labels. No accuracy, precision, recall or F1 is computed here from another model or the rules. A future verified-human evaluation must report unreviewed, unverified, unresolved and abstained cases and use sampling weights appropriately.

In [ ]:
HUMAN_REVIEW_PATH = PREPARED / 'independent_human_review.csv'
human = pd.read_csv(HUMAN_REVIEW_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
assert human.advertisement_id.is_unique
assert set(human.advertisement_id) == set(sample.advertisement_id)
entered = human.human_relevance.str.strip().ne('')
verified = human.human_verified.str.casefold().eq('true') & human.reviewer.str.strip().ne('')
unresolved_human = verified & (human.human_relevance.eq('uncertain') | human.human_relevance.eq(''))
display(pd.DataFrame([{'total_review_rows': len(human), 'unreviewed': int((~entered).sum()),
    'entered_but_unverified': int((entered & ~verified).sum()), 'explicitly_verified': int(verified.sum()),
    'verified_unresolved_relevance': int(unresolved_human.sum())}]))
print('No human correctness metrics calculated. Review file:', HUMAN_REVIEW_PATH)